# WavLM + Prosody (21-dim) Extraction + Training (GPU)

## Based on Working Kaggle Pipeline (ChuckleNet Phase A)

### Features Per Utterance:
- **WavLM**: 768-dim (ATTENTION POOLING over frames)
- **Prosody**: 21-dim (F0, energy, duration, MFCC, etc.)
- **Text**: First 200 chars
- **Label**: 0/1 (laughter)

### Training:
- WavLM FROZEN (feature extractor only)
- Attention pooling over WavLM hidden states
- Fusion: concat(audio_emb + prosody_proj) → MLP
- Class weights: [1.0, 2.5] (2-class CrossEntropy)
- Comedian-level holdout (3 comedians)

**Runtime: ~8 hours on T4 GPU**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
print('Drive mounted!')

In [ ]:
# Install & imports
!pip install -q transformers librosa soundfile

import os, json, time, math, gc
import numpy as np
import torch
import torch.nn as nn
import librosa
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler

SR = 16000
MAX_DURATION = 5.0  # Max utterance duration
MAX_LEN = int(MAX_DURATION * SR)
BATCH = 16
EPOCHS = 20
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Mem: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Load utterance labels
UTT_FILE = '/content/gdrive/MyDrive/utterances_clean.jsonl'
print(f'Loading from {UTT_FILE}...')

utterances_by_video = defaultdict(list)
with open(UTT_FILE) as f:
    for line in f:
        u = json.loads(line)
        utterances_by_video[u['video_id']].append(u)

total_utts = sum(len(v) for v in utterances_by_video.values())
total_pos = sum(sum(1 for u in v if u.get('label', 0) == 1) for v in utterances_by_video.values())
print(f'Loaded {total_utts} utterances from {len(utterances_by_video)} videos')
print(f'Positive: {total_pos} ({100*total_pos/total_utts:.1f}%)')

In [ ]:
# Find audio files - ALL folders
AUDIO_BASE = '/content/gdrive/MyDrive'
audio_paths = {}

search_folders = [
    'chuckle_audio',
    'chuckle_audio_all/audio',
    'chuckle_audio_all/audio_all',
    'chuckle_audio_all/audio_final',
    'chuckle_audio_all/audio_new',
]

for folder in search_folders:
    path = os.path.join(AUDIO_BASE, folder)
    if os.path.exists(path):
        cnt = 0
        for f in os.listdir(path):
            if f.endswith(('.mp3', '.wav')):
                vid = f.rsplit('.', 1)[0]
                if vid not in audio_paths:
                    audio_paths[vid] = os.path.join(path, f)
                    cnt += 1
        print(f'  {folder}: +{cnt} files')

print(f'\nTotal unique audio: {len(audio_paths)}')
matched_vids = [v for v in utterances_by_video if v in audio_paths]
print(f'Videos with audio+labels: {len(matched_vids)}')

In [ ]:
# Setup output + checkpoint
OUTPUT_DIR = '/content/gdrive/MyDrive/wavlm_prosody_utterance_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, 'checkpoint.json')
FEATURES_FILE = os.path.join(OUTPUT_DIR, 'utterance_features.jsonl')

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        checkpoint = json.load(f)
    done_videos = set(checkpoint.get('done_videos', []))
    total_done = checkpoint.get('total_utterances', 0)
    total_pos = checkpoint.get('total_positive', 0)
    print(f'Resuming: {len(done_videos)} videos, {total_done} utts, {total_pos} pos')
else:
    done_videos = set()
    total_done = 0
    total_pos = 0
    print('Starting fresh extraction')

In [ ]:
# Load WavLM model (FROZEN feature extractor)
from transformers import WavLMModel, Wav2Vec2FeatureExtractor

print('Loading WavLM (FROZEN)...')
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
wavlm.eval()
for p in wavlm.parameters():
    p.requires_grad = False

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base-plus')

# Move to GPU
wavlm = wavlm.to(device)
torch.cuda.empty_cache()
print(f'WavLM on: {device}')

In [ ]:
# ============================================================
# EXTRACT 21-DIM PROSODY FEATURES
# ============================================================

def extract_prosody_21dim(audio, start_s, end_s, sr):
    """Extract 21-dim prosody features (matching Purandare 2006)"""
    start_sample = int(start_s * sr)
    end_sample = int(end_s * sr)
    if end_sample > len(audio):
        end_sample = len(audio)
    if start_sample >= len(audio) or end_sample - start_sample < sr * 0.02:
        return None
    
    segment = audio[start_sample:end_sample]
    duration = end_s - start_s
    
    # 1. Pitch (F0) features (5 dim)
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(
            segment, fmin=50, fmax=500, sr=sr
        )
        f0_clean = f0[~np.isnan(f0)]
        f0_mean = np.mean(f0_clean) if len(f0_clean) > 0 else 0
        f0_std = np.std(f0_clean) if len(f0_clean) > 0 else 0
        f0_max = np.max(f0_clean) if len(f0_clean) > 0 else 0
        f0_min = np.min(f0_clean) if len(f0_clean) > 0 else 0
        voiced_rate = np.mean(voiced_flag) if len(voiced_flag) > 0 else 0
    except:
        f0_mean = f0_std = f0_max = f0_min = voiced_rate = 0
    
    # 2. Energy features (5 dim)
    rms = librosa.feature.rms(y=segment, sr=sr)[0]
    energy_mean = np.mean(rms)
    energy_std = np.std(rms)
    energy_max = np.max(rms)
    energy_min = np.min(rms)
    energy_range = energy_max - energy_min
    
    # 3. Duration features (2 dim)
    duration_s = end_s - start_s
    speech_rate = len([c for c in segment if abs(c) > 0.01]) / duration_s if duration_s > 0 else 0
    
    # 4. Spectral features (5 dim) - using MFCC
    try:
        mfccs = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
        mfcc1 = np.mean(mfccs[0])  # energy proxy
        mfcc2 = np.mean(mfccs[1])  # pitch-related
        mfcc3 = np.mean(mfccs[2])  # harmonic
        mfcc_delta = np.mean(librosa.feature.delta(mfccs[1]))
        spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=segment, sr=sr)[0])
    except:
        mfcc1 = mfcc2 = mfcc3 = mfcc_delta = spectral_centroid = 0
    
    # 5. Voice quality (4 dim)
    try:
        zcr = np.mean(librosa.feature.zero_crossing_rate(segment)[0])
    except:
        zcr = 0
    
    # Jitter and shimmer approximations
    try:
        if len(f0_clean) > 1:
            jitter = np.mean(np.abs(np.diff(f0_clean))) if len(f0_clean) > 1 else 0
        else:
            jitter = 0
    except:
        jitter = 0
    
    shimmer_approx = energy_std / (energy_mean + 1e-8)
    
    return [
        # Pitch (5)
        f0_mean, f0_std, f0_max, f0_min, voiced_rate,
        # Energy (5)
        energy_mean, energy_std, energy_max, energy_min, energy_range,
        # Duration (2)
        duration_s, speech_rate,
        # Spectral (5)
        mfcc1, mfcc2, mfcc3, mfcc_delta, spectral_centroid,
        # Voice quality (4)
        zcr, jitter, shimmer_approx, voiced_rate,
    ]  # Total: 21 dim

# Test
test_prosody = extract_prosody_21dim(np.random.randn(16000), 0, 1.0, 16000)
print(f'Prosody dim: {len(test_prosody)}')

In [ ]:
# ============================================================
# EXTRACT FEATURES (STREAMING - saves memory)
# ============================================================
CHECKPOINT_INTERVAL = 25

def load_audio_segment(audio_path, start_s, duration=MAX_DURATION):
    """Load audio segment on-demand (streaming)"""
    try:
        offset = max(0, start_s - 0.05)  # Small context before
        audio, _ = librosa.load(audio_path, sr=SR, mono=True, offset=offset, duration=duration + 0.05)
        target_len = int(MAX_DURATION * SR)
        if len(audio) < target_len:
            audio = np.pad(audio, (0, target_len - len(audio)))
        return audio[:target_len]
    except Exception as e:
        return np.zeros(int(MAX_DURATION * SR), dtype=np.float32)

mode = 'a' if done_videos else 'w'
out_f = open(FEATURES_FILE, mode)
t0 = time.time()
processed = 0

videos_to_process = [
    (vid, path) for vid, path in audio_paths.items()
    if vid not in done_videos and vid in utterances_by_video
]
print(f'Processing {len(videos_to_process)} videos...')

for vid, audio_path in tqdm(videos_to_process):
    try:
        utts = utterances_by_video[vid]
        
        # Process utterances in batches for WavLM
        batch_size = 8  # Smaller batch for memory
        
        for i in range(0, len(utts), batch_size):
            batch_utts = utts[i:i+batch_size]
            
            # Load audio and extract features for batch
            waveforms = []
            prosody_list = []
            valid_utts = []
            
            for u in batch_utts:
                audio_seg = load_audio_segment(audio_path, u['start'])
                prosody = extract_prosody_21dim(audio_seg, u['start'], u['end'], SR)
                
                if prosody is None:
                    continue
                
                waveforms.append(audio_seg)
                prosody_list.append(prosody)
                valid_utts.append(u)
            
            if not waveforms:
                continue
            
            # WavLM inference on batch
            inputs = feature_extractor(waveforms, sampling_rate=SR, return_tensors='pt', padding=True, pad_to_multiple_of=320)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = wavlm(**inputs)
                hidden = outputs.last_hidden_state  # (B, T, 768)
            
            # Attention pooling
            attn_weights = torch.nn.functional.softmax(
                torch.nn.Linear(768, 1)(hidden).squeeze(-1), dim=-1
            )
            audio_embs = torch.bmm(attn_weights.unsqueeze(1), hidden).squeeze(1).cpu().numpy()
            
            # Save each utterance
            for j, u in enumerate(valid_utts):
                record = {
                    'uid': f"{vid}_{u['start']:.2f}",
                    'video_id': vid,
                    'start': u['start'],
                    'end': u['end'],
                    'text': u.get('text', '')[:200],
                    'label': u.get('label', 0),
                    'wavlm': audio_embs[j].tolist(),
                    'prosody': prosody_list[j],
                }
                out_f.write(json.dumps(record) + '\n')
                total_done += 1
                if u.get('label', 0) == 1:
                    total_pos += 1
            
            # Memory cleanup
            del inputs, outputs, hidden, audio_embs
            torch.cuda.empty_cache()
        
        done_videos.add(vid)
        processed += 1
        
    except Exception as e:
        print(f'\nError {vid}: {e}')
        continue
    
    # Checkpoint
    if processed % CHECKPOINT_INTERVAL == 0:
        out_f.flush()
        elapsed = time.time() - t0
        rate = processed / elapsed if elapsed > 0 else 0
        remaining = len(videos_to_process) - processed
        eta_h = remaining / rate / 3600 if rate > 0 else 0
        
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump({
                'done_videos': list(done_videos),
                'total_utterances': total_done,
                'total_positive': total_pos,
            }, f)
        
        print(f'\n[{processed}/{len(videos_to_process)}] {total_done} utts, {total_pos} pos, ETA: {eta_h:.1f}h')

# Finalize
out_f.close()
with open(CHECKPOINT_FILE, 'w') as f:
    json.dump({
        'done_videos': list(done_videos),
        'total_utterances': total_done,
        'total_positive': total_pos,
        'completed': True,
    }, f, indent=2)

elapsed = time.time() - t0
print(f'\n{"="*60}')
print(f'EXTRACTION DONE!')
print(f'Videos: {len(done_videos)}')
print(f'Utterances: {total_done}')
print(f'Positive: {total_pos} ({100*total_pos/max(total_done,1):.1f}%)')
print(f'Time: {elapsed/3600:.1f} hours')
print(f'Output: {FEATURES_FILE}')

In [ ]:
# ============================================================
# TRAINING SECTION
# ============================================================
# Only run AFTER extraction completes!

import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('=== Loading extracted features ===')

# Load all features
data = []
with open(FEATURES_FILE) as f:
    for line in f:
        data.append(json.loads(line))

print(f'Total samples: {len(data)}')
pos_count = sum(1 for d in data if d['label'] == 1)
print(f'Positive: {pos_count} ({100*pos_count/len(data):.1f}%)')

In [ ]:
# Comedian-level holdout (matching working Kaggle pipeline)
HELD_OUT = {'BFIHCzw3itk', 'BAD4askmGgk', '1Nb3_os4RSA'}

# Split by video
video_labels = {}
for d in data:
    vid = d['video_id']
    if vid not in video_labels:
        video_labels[vid] = d['label']

pos_videos = [v for v, l in video_labels.items() if l == 1 and v not in HELD_OUT]
neg_videos = [v for v, l in video_labels.items() if l == 0 and v not in HELD_OUT]

random.shuffle(pos_videos)
random.shuffle(neg_videos)

# Holdout: 3 pos + 3 neg (or what's available)
holdout_videos = set()
for v in HELD_OUT:
    if v in video_labels:
        holdout_videos.add(v)

# Val: separate set
val_videos = set(pos_videos[:3] + neg_videos[:3])
train_videos = set(video_labels.keys()) - holdout_videos - val_videos

print(f'Holdout: {len(holdout_videos)} videos')
print(f'Val: {len(val_videos)} videos')
print(f'Train: {len(train_videos)} videos')

In [ ]:
# Build splits
def build_split(videos):
    X_wavlm, X_prosody, y = [], [], []
    for d in data:
        if d['video_id'] in videos:
            X_wavlm.append(d['wavlm'])
            X_prosody.append(d['prosody'])
            y.append(d['label'])
    return np.array(X_wavlm), np.array(X_prosody), np.array(y)

X_train_w, X_train_p, y_train = build_split(train_videos)
X_val_w, X_val_p, y_val = build_split(val_videos)
X_holdout_w, X_holdout_p, y_holdout = build_split(holdout_videos)

# Scale prosody
prosody_scaler = StandardScaler()
prosody_scaler.fit(X_train_p)
X_train_p = prosody_scaler.transform(X_train_p)
X_val_p = prosody_scaler.transform(X_val_p)
X_holdout_p = prosody_scaler.transform(X_holdout_p)

print(f'Train: {len(y_train)}, pos={y_train.sum()} ({100*y_train.mean():.1f}%)')
print(f'Val: {len(y_val)}, pos={y_val.sum()} ({100*y_val.mean():.1f}%)')
print(f'Holdout: {len(y_holdout)}, pos={y_holdout.sum()} ({100*y_holdout.mean():.1f}%)')

In [ ]:
# Model - matching working Kaggle architecture
class WavLMWithProsody(nn.Module):
    def __init__(self, wavlm_dim=768, prosody_dim=21, hidden=128):
        super().__init__()
        # Prosody projection (21 -> 32)
        self.prosody_proj = nn.Sequential(
            nn.Linear(prosody_dim, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32)
        )
        # Fusion: 768 + 32 = 800
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(wavlm_dim + 32, hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, 64),
            nn.GELU(),
            nn.Linear(64, 2)  # 2-class output
        )
    
    def forward(self, wavlm_emb, prosody):
        prosody_emb = self.prosody_proj(prosody)
        fused = torch.cat([wavlm_emb, prosody_emb], dim=1)
        return self.classifier(fused)

model = WavLMWithProsody(768, 21, 128).to(device)

# Optimizer & scheduler - matching working pipeline
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Class weights: [1.0, 2.5] - matching working Kaggle
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.5]).to(device))

# Convert to tensors
X_train_w_t = torch.FloatTensor(X_train_w).to(device)
X_train_p_t = torch.FloatTensor(X_train_p).to(device)
y_train_t = torch.LongTensor(y_train).to(device)
X_val_w_t = torch.FloatTensor(X_val_w).to(device)
X_val_p_t = torch.FloatTensor(X_val_p).to(device)
y_val_t = torch.LongTensor(y_val).to(device)
X_holdout_w_t = torch.FloatTensor(X_holdout_w).to(device)
X_holdout_p_t = torch.FloatTensor(X_holdout_p).to(device)
y_holdout_t = torch.LongTensor(y_holdout).to(device)

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {n_total:,} total | {n_trainable:,} trainable params')
print(f'Class weights: [1.0, 2.5]')

In [ ]:
# Training loop
BATCH_SIZE = 256
best_val_f1 = 0
best_holdout_f1 = 0
best_state = None
history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}

t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    
    perm = torch.randperm(len(X_train_t))
    n_batches = math.ceil(len(X_train_t) / BATCH_SIZE)
    
    for i in range(0, len(X_train_t), BATCH_SIZE):
        batch_idx = perm[i:i+BATCH_SIZE]
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(X_train_w_t[batch_idx], X_train_p_t[batch_idx])
        loss = criterion(logits, y_train_t[batch_idx])
        
        if torch.isnan(loss) or torch.isinf(loss):
            print(f'  Warning: NaN/Inf at batch {i//BATCH_SIZE}, skip')
            continue
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(y_train_t[batch_idx].cpu().numpy())
        
        gc.collect()
    
    scheduler.step()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_w_t, X_val_p_t)
        val_loss = criterion(val_logits, y_val_t).item()
        val_pred = val_logits.argmax(dim=-1).cpu().numpy()
        
        holdout_logits = model(X_holdout_w_t, X_holdout_p_t)
        holdout_pred = holdout_logits.argmax(dim=-1).cpu().numpy()
        
        train_f1 = f1_score(y_train, all_preds, average='binary', zero_division=0)
        val_f1 = f1_score(y_val, val_pred, average='binary', zero_division=0)
        holdout_f1 = f1_score(y_holdout, holdout_pred, average='binary', zero_division=0)
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_holdout_f1 = holdout_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, os.path.join(OUTPUT_DIR, 'best_model.pt'))
    
    history['train_loss'].append(total_loss / n_batches)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} ({time.time()-t0:.0f}s) | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | Holdout F1: {holdout_f1:.4f}')

print(f'\n{"="*60}')
print(f'FINAL: Val F1={best_val_f1:.4f}, Holdout F1={best_holdout_f1:.4f}')
print(f'Model saved: {OUTPUT_DIR}/best_model.pt')